# Validate live-loop fixes on real captured chains

Proves **Buckets 1-3** (option-type routing, fill reconciliation, real MTM P&L) using the
**real repo functions** on the **captured SPY chains** from the `live-data` branch.

Run top-to-bottom on Colab. The Heston calibration needs JAX/Diffrax, which segfaults on the
Windows dev box — hence this notebook. Each section ends with `assert`ed pass criteria; a
failing assert stops the notebook.

In [ ]:
# --- Setup: clone the fix branch, install deps, fetch the captured chains ---
import os, subprocess, sys
BRANCH = "fix/option-type-reconciliation-mtm"   # push this branch first if not yet on GitHub
REPO   = "https://github.com/AliAlpOezer/heston-arb.git"
if not os.path.isdir("heston-arb"):
    try:
        subprocess.run(["git","clone","--branch",BRANCH,REPO], check=True)
    except subprocess.CalledProcessError:
        print(f"WARNING: branch {BRANCH} not on remote -> cloning master. "
              "Push the branch first, or the fixes will be ABSENT.")
        subprocess.run(["git","clone",REPO], check=True)
os.chdir("heston-arb")
subprocess.run(["git","fetch","origin","live-data"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q",
                "diffrax","optax","equinox","pandas","pyarrow","scipy","cvxpy"], check=False)
print("cwd:", os.getcwd())
print("HEAD:", subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"],
                              capture_output=True,text=True).stdout.strip())

In [ ]:
# --- Reconstruct an OptionsChain from a captured snapshot ---
import pandas as pd, numpy as np, subprocess
DATE = "2026-06-25"   # a captured trading day on the live-data branch
blob = subprocess.run(["git","show",f"origin/live-data:chains/SPY/{DATE}.parquet"],
                      capture_output=True).stdout
open(f"/tmp/{DATE}.parquet","wb").write(blob)
df = pd.read_parquet(f"/tmp/{DATE}.parquet")
t0 = sorted(df["time"].unique())[0]
snap = df[df["time"] == t0]
from data.cleaner import OptionsChain
chain = OptionsChain(
    ticker="SPY", snapshot_time=str(t0),
    spot=float(snap["spot"].iloc[0]), r=float(snap["r"].iloc[0]), q=float(snap["q"].iloc[0]),
    strikes=snap["strike"].to_numpy(float), maturities=snap["maturity"].to_numpy(float),
    mid_prices=snap["mid"].to_numpy(float), bid_prices=snap["bid"].to_numpy(float),
    ask_prices=snap["ask"].to_numpy(float), open_interest=snap["oi"].to_numpy(float),
    option_type=snap["opt_type"].to_numpy(str),
)
print(f"snapshot {t0}  spot={chain.spot:.2f}  n={len(chain.strikes)} "
      f"(calls {int(np.sum(chain.option_type=='C'))}, puts {int(np.sum(chain.option_type=='P'))})")
print("call strikes:", int(chain.strikes[chain.option_type=='C'].min()), "-",
      int(chain.strikes[chain.option_type=='C'].max()),
      "| put strikes:", int(chain.strikes[chain.option_type=='P'].min()), "-",
      int(chain.strikes[chain.option_type=='P'].max()))

In [ ]:
# --- Real pipeline: clean -> surface -> calibrate (JAX) -> detect mispricings ---
import warnings, config
from data.cleaner import clean_chain
from data.surface import build_surface
from calibration.calibrator import calibrate, calibration_rmse
from signals.mispricing import detect_mispricings
from trading.loop import _build_cal_input
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cleaned = clean_chain(chain)
surface = build_surface(cleaned)
cal = _build_cal_input(surface, chain)
fitted, _loss = calibrate(cal, n_steps=300)
rmse = calibration_rmse(fitted, cal)
sm = detect_mispricings(surface, fitted, rmse)
feller = bool(2*fitted.kappa*fitted.theta >= fitted.xi**2)
print(f"params: kappa={fitted.kappa:.3f} theta={fitted.theta:.4f} xi={fitted.xi:.3f} "
      f"rho={fitted.rho:.3f} v0={fitted.v0:.4f}")
print(f"RMSE={rmse:.4f}  Feller={'OK' if feller else 'VIOLATED'}  signals={len(sm.signals)}")
assert feller, "Feller must hold on the fit"


## Bucket 1 - option-type routing
The IV surface merges OTM puts (low strikes) and OTM calls (high strikes). The OLD loop sent
**every** signal as a CALL (`right="C"`), so a put-wing strike (K<F) became a deep-ITM call
priced off the cheap OTM put (~$0.04) — unfillable. The fix routes by `right = "P" if K<F else "C"`.

In [ ]:
# --- Bucket 1: where signals land, and what we now trade vs. what the old code did ---
from trading.loop import _nbbo_mid
S = chain.spot
print(f"{'strike':>7} {'fwd':>7} {'dir':>4} | NEW {'instr':>5} {'limit':>7} | "
      f"OLD instr=C {'limit':>7} {'callITM':>8}")
n_putwing = n_putrouted = 0
for s in sm.signals[:25]:
    F = float(chain.forward(s.maturity))
    right = "P" if s.strike < F else "C"                       # NEW routing (the fix)
    new_px = _nbbo_mid(chain, s.strike, s.maturity, opt_type=right)
    old_px = _nbbo_mid(chain, s.strike, s.maturity)            # OLD: no opt_type filter
    call_itm = max(S - s.strike, 0.0)                          # intrinsic of the call OLD ordered
    if s.strike < F:
        n_putwing += 1
        n_putrouted += int(right == "P")
    npx = f"{new_px:.3f}" if new_px else "None"
    opx = f"{old_px:.3f}" if old_px else "None"
    print(f"{s.strike:7.0f} {F:7.1f} {s.direction:>4} |    {right:>5} {npx:>7} | "
          f"        {opx:>7} {call_itm:8.1f}")
print(f"\nput-wing signals: {n_putwing}  ->  routed to PUT: {n_putrouted}")

# Pass criteria
assert n_putwing == 0 or n_putrouted == n_putwing, "every put-wing signal must route to a PUT"
for s in sm.signals[:25]:
    if s.strike < float(chain.forward(s.maturity)):
        px = _nbbo_mid(chain, s.strike, s.maturity, opt_type="P")
        assert px is not None and px > 0, "OTM put must have a real, fillable price"
        assert (S - s.strike) > px*10, "OLD call (limit=put px) was deep-ITM -> never fills"
print("Bucket 1 PASS: put-wing signals now trade real OTM puts at fillable prices; the old code "
      "sent deep-ITM CALLs at the put's price (never fill).")

## Bucket 2 - fill reconciliation
Reconciliation drops never-filled phantoms, keeps held positions (syncing qty + the real fill price), and resets the hedge to the broker's actual share count.

In [ ]:
# --- Bucket 2: reconcile against a mock broker ---
from trading.broker import Position, Order, OptionContract, _occ_symbol
from trading.state import LiveState, LivePosition
from trading.loop import _reconcile_positions
occ_held = _occ_symbol(OptionContract("SPY","2026-06-30",605.0,"P"))
class MockBroker:
    def get_positions(self):
        return [Position(occ_held, 7, avg_entry_price=0.037, asset_class="option"),
                Position("SPY", 1, asset_class="equity")]
    def get_open_orders(self):
        return [Order("OPEN-1","x","BUY",1,"LMT",0.04,"")]
def pos(strike, right, oid):
    return LivePosition(ticker="SPY", entry_date="2026-06-25", entry_time="t", strike=strike,
                        maturity=0.0137, expiry="2026-06-30", right=right, direction="buy", qty=5,
                        entry_market_iv=0.59, entry_model_iv=0.61, entry_vol_gap=0.05,
                        entry_spot=734.0, entry_premium=28.0, entry_fill_price=0.04,
                        filled=False, option_order_id=oid)
st = LiveState(current_hedge=-10000)
st.open_positions = [pos(605.0,"P","OLD-1"), pos(610.0,"C","EXP-1")]   # held, phantom
res = _reconcile_positions(st, MockBroker(), "SPY", "2026-06-26", verbose=True)
held, phantom = st.open_positions
assert held.filled and held.qty == 7 and abs(held.entry_fill_price-0.037) < 1e-9
assert phantom.exited and phantom.exit_reason == "unfilled"
assert st.current_hedge == 1
print("\nBucket 2 PASS:", res)

## Bucket 3 - real mark-to-market P&L
P&L is `(mark - entry_fill) x qty x mult` (a level), not the old vega term re-added every tick.

In [ ]:
# --- Bucket 3: MTM signs + no tick-count drift ---
MULT = config.CONTRACT_MULTIPLIER
def mtm(direction, qty, entry, mark):
    sign = 1.0 if direction == "buy" else -1.0
    return sign * qty * MULT * (mark - entry)
assert mtm("buy", 7, 0.04, 0.10) == 7*MULT*0.06            # long gains when price rises
assert mtm("sell", 7, 0.10, 0.04) == 7*MULT*0.06           # short gains when price falls
levels = [mtm("buy", 7, 0.04, 0.05) for _ in range(100)]   # constant mark, 100 ticks
assert len(set(levels)) == 1, "constant mark must give a constant P&L level (no drift)"
print(f"Bucket 3 PASS: signs correct; 100 ticks at a constant mark -> P&L stays "
      f"{levels[0]:+.2f} (old vega-accumulation drifted ~x100).")

## Summary
If every cell ran without an `AssertionError`:
- **Bucket 1** — put-wing signals route to real, fillable OTM puts (not deep-ITM calls).
- **Bucket 2** — reconciliation drops phantoms, keeps and syncs held positions, resets the hedge to reality.
- **Bucket 3** — P&L is real mark-to-market and no longer drifts with tick count.

Still open before re-enabling live: **Bucket 4** (tighten the moneyness gate; assess whether the
put-wing skew gaps are real edge or Heston fit error; add stop-loss / take-profit) and a live
`get_positions` smoke test against Alpaca.